In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import AutoMinorLocator
from scipy import stats

# ── Load data ─────────────────────────────────────────────────────────────────
file = r"/home/nithin/eda/LPCAS_TTSKY26a/tb_Gm_Cell_v2_MC_Sims_TB.txt"
df   = pd.read_csv(file, sep=r'\s+', header=None)

run      = df[1].values   # column 2 → Run number
mismatch = df[3].values   # column 4 → mismatch value

# ── Reusable plot function ────────────────────────────────────────────────────
def plot_mc_histogram(data, signal_label, unit='', unit_scale=1.0, unit_suffix='',
                      save_path=None):
    """
    unit_scale : multiply raw data for display  (e.g. 1e3 to show mA instead of A)
    unit_suffix: string appended to axis label  (e.g. 'mA', 'mV', 'µA/V')
    """
    d    = data * unit_scale
    mean = np.mean(d)
    std  = np.std(d, ddof=1)
    var  = np.var(d, ddof=1)
    n    = len(d)

    BG        = '#1a1a2e'
    PANEL_BG  = '#16213e'
    BAR_COLOR = '#4fc3f7'
    BAR_EDGE  = '#0288d1'
    CURVE_CLR = '#ff6f00'
    MEAN_CLR  = '#ffffff'
    SIG1_CLR  = '#76ff03'
    SIG2_CLR  = '#ffeb3b'
    SIG3_CLR  = '#ff5252'
    TEXT_CLR  = '#e0e0e0'
    GRID_CLR  = '#2a3a5a'

    fig = plt.figure(figsize=(12, 7), facecolor=BG)
    ax  = fig.add_subplot(111, facecolor=PANEL_BG)

    num_bins = int(np.ceil(np.sqrt(n)))
    counts, bin_edges, _ = ax.hist(
        d, bins=num_bins,
        color=BAR_COLOR, edgecolor=BAR_EDGE, linewidth=0.6,
        alpha=0.85, zorder=2, label='Histogram'
    )

    x_fit      = np.linspace(d.min() - 3*std, d.max() + 3*std, 600)
    bin_width  = bin_edges[1] - bin_edges[0]
    pdf_scaled = stats.norm.pdf(x_fit, mean, std) * n * bin_width
    ax.plot(x_fit, pdf_scaled, color=CURVE_CLR, linewidth=2.5, zorder=5,
            label='Gaussian fit')

    y_max = counts.max()

    sigma_specs = [(3, SIG3_CLR, 0.12), (2, SIG2_CLR, 0.18), (1, SIG1_CLR, 0.22)]
    for k, col, alpha in sigma_specs:
        ax.axvspan(mean - k*std, mean + k*std,
                   ymin=0, ymax=1, color=col, alpha=alpha, zorder=1)
    for k, col, _ in sigma_specs:
        for side in [-1, 1]:
            ax.axvline(mean + side*k*std, color=col, linewidth=1.0,
                       linestyle='--', alpha=0.85, zorder=4)

    ax.axvline(mean, color=MEAN_CLR, linewidth=2.2, linestyle='-',
               zorder=6, label=f'Mean = {mean:.4f} {unit_suffix}')

    bracket_y_frac = [0.88, 0.76, 0.64]
    for (k, col, _), frac in zip(sigma_specs[::-1], bracket_y_frac):
        y_br = y_max * frac
        ax.annotate('', xy=(mean + k*std, y_br), xytext=(mean - k*std, y_br),
                    arrowprops=dict(arrowstyle='<->', color=col, lw=1.5))
        ax.text(mean, y_br + y_max*0.015,
                f'±{k}σ = ±{k*std:.4f} {unit_suffix}',
                ha='center', va='bottom', fontsize=8, color=col,
                fontfamily='monospace')

    stats_lines = [
        f"N       = {n}",
        f"Mean    = {mean:.6f} {unit_suffix}",
        f"σ       = {std:.6f} {unit_suffix}",
        f"σ/μ     = {abs(std/mean)*100:.3f} %",
        f"σ²      = {var:.3e}",
        f"+3σ     = {mean+3*std:.6f} {unit_suffix}",
        f"−3σ     = {mean-3*std:.6f} {unit_suffix}",
        f"3σ span = {6*std:.6f} {unit_suffix}",
    ]
    ax.text(0.017, 0.97, '\n'.join(stats_lines), transform=ax.transAxes,
            fontsize=8.5, verticalalignment='top', color=TEXT_CLR,
            fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#0d1b2a',
                      edgecolor='#4fc3f7', alpha=0.92), zorder=10)

    p1 = mpatches.Patch(color=SIG1_CLR, alpha=0.55, label='±1σ (68.27 %)')
    p2 = mpatches.Patch(color=SIG2_CLR, alpha=0.55, label='±2σ (95.45 %)')
    p3 = mpatches.Patch(color=SIG3_CLR, alpha=0.55, label='±3σ (99.73 %)')
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [p1, p2, p3], loc='upper right', fontsize=8.5,
              facecolor='#0d1b2a', edgecolor='#4fc3f7', labelcolor=TEXT_CLR)

    ax.set_xlabel(f'{signal_label}  ({unit_suffix})', color=TEXT_CLR, fontsize=11, labelpad=6)
    ax.set_ylabel('Count',                            color=TEXT_CLR, fontsize=11, labelpad=6)
    ax.set_title(f'Monte Carlo — {signal_label} Mismatch Distribution  |  tb_Gm_Cell_v2',
                 color=TEXT_CLR, fontsize=13, fontweight='bold', pad=14)

    ax.tick_params(colors=TEXT_CLR, which='both', direction='in', top=True, right=True)
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    for spine in ax.spines.values():
        spine.set_edgecolor('#3a5070')
    ax.grid(True, which='major', color=GRID_CLR, linewidth=0.6, zorder=0)
    ax.grid(True, which='minor', color=GRID_CLR, linewidth=0.3, linestyle=':', zorder=0)
    ax.set_xlim(d.min() - 2.5*std, d.max() + 2.5*std)
    ax.set_ylim(0, y_max * 1.18)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=180, bbox_inches='tight', facecolor=BG)
        print(f"Saved → {save_path}")
    plt.show()

# ── Plot ──────────────────────────────────────────────────────────────────────
# Adjust unit_scale and unit_suffix to match your signal's unit
# e.g. if mismatch is in A → use unit_scale=1e6, unit_suffix='µA'
#      if mismatch is in V → use unit_scale=1e3, unit_suffix='mV'
plot_mc_histogram(mismatch,
                  signal_label='Gm Mismatch',
                  unit_scale=1e6,        # ← change if needed
                  unit_suffix='µA/V',    # ← change to match your signal
                  save_path='/home/nithin/eda/LPCAS_TTSKY26a/mc_hist_mismatch.png')